# Segmentación de objetos en videos con SAM 2

En este laboratorio entenderemos cómo usar SAM 2 para la segmentación interactiva en videos. Incluyendo lo siguiente:

- agregar clics en un fotograma para obtener y refinar _masklets_ (máscaras espacio-temporales)
- propagar clics para obtener _masklets_ a lo largo del video

Utilizaremos los términos _segmento_ o _máscara_ para referirnos a la predicción del modelo para un objeto en un solo cuadro, y _masklet_ para referirnos a las máscaras espacio-temporales en todo el video.


Segment Anything Model 2 (SAM 2) predice máscaras de objetos a partir de prompts que indican el objeto deseado. El modelo primero convierte la imagen en una representación de la imagen que permite producir máscaras de alta calidad de manera eficiente a partir de un prompt.

La clase `SAM2ImagePredictor` proporciona una interfaz sencilla del modelo para proveer prompts. Permite al usuario establecer primero una imagen utilizando el método `set_image`, que calcula las representaciones de imagen necesarias. Luego, se pueden proporcionar prompts a través del método `predict` para predecir de manera eficiente las máscaras a partir de ellos. El modelo puede tomar como entrada puntos o cuadros envolventes, así como segmentos de la iteración anterior de inferencia.

## Primero ingresa tu código uniandes

In [ ]:
codigo = None #CODIGO UNIANDES

## Importación de librerías
Empezaremos importando las librerías necesarias para correr nuestro laboratorio. Adicionalmente, definiremos el `device` que será utilizado a lo largo del labortorio. En este caso, utilizaremos `cuda`. 

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import random
from PIL import Image
import sam2
import lm

# Selección del device
if torch.cuda.is_available():
    device = torch.device("cuda")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## Funciones auxiliares

A continuación, crearemos las funciones auxiliares necesarias para visualizar diferentes elementos clave en nuestro proceso de segmentación. Estas funciones nos ayudarán a representar de manera visual los puntos de interés y las máscaras sobre el video, facilitando la interpretación de los resultados.

* `show_mask`: Superpone una máscara de segmentación sobre la imagen, utilizando un color semitransparente para resaltar las áreas correspondientes.

* `show_points`: Dibuja puntos positivos y negativos sobre la imagen, representando visualmente las áreas de interés clave en el proceso de segmentación.

In [ ]:
np.random.seed(3)

def show_mask(mask, ax, obj_id=None, random_color=False):
    mask = np.array(mask)
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

## Selección de objetos con SAM 2

Primero, cargamos el modelo y predictor de SAM 2. Para esto utilizaremos la función `build_sam2_video_predictor` de la libreria de sam2. 

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"
predictor = build_sam2_video_predictor(model_cfg, checkpoint)

## Vídeo de ejemplo

Asumamos que el vídeo se almacena como una lista de fotogramas JPEG con nombres de archivo como `<frame_index>.jpg`. Para sus vídeos personalizados, puede extraer sus fotogramas JPEG con ffmpeg (https://ffmpeg.org/) de la siguiente manera:
```
ffmpeg -i <your_video>.mp4 -q:v 2 -start_number 0 <output_dir>/'%05d.jpg'
```
donde `-q:v` genera fotogramas JPEG de alta calidad y `-start_number 0` le pide a ffmpeg que inicie el archivo JPEG desde `00000.jpg`.

In [ ]:
# `video_dir` una carpeta que contiene las imágenes de los frames del video
video_dir = "./videos/video_000"

# escanear todas las imágenes dentro de la carpeta
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

# Veamos el primer fotograma
frame_idx = 0
plt.figure(figsize=(9, 6))
plt.title(f"frame {frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[frame_idx])))
plt.axis("off")

## Inicializar el estado de inferencia
SAM 2 requiere realizar inferencia con estado para la segmentación de video interactivo, por lo que necesitamos inicializar un **estado de inferencia** en este video. Durante la inicialización, el modelo carga todos los fotogramas JPEG en `video_path` y almacena sus píxeles en `inference_state` (como se muestra en la barra de progreso a continuación).

In [ ]:
inference_state = predictor.init_state("./videos/video_000")

## Segmentar y seguir un objeto

### Paso 1: Agregar un primer clic en un fotograma
Para comenzar, intentemos segmentar la estatua del bobo. Para lograr esto, debemos hacer un **clic positivo** en (x, y) = (210, 350) con la etiqueta `1`, enviando sus coordenadas y etiquetas a la API `add_new_points_or_box`.

Nota: la etiqueta `1` indica un *clic positivo (para agregar una región)*, mientras que la etiqueta `0` indica un *clic negativo (para eliminar una región)*.

In [ ]:
ann_frame_idx = 0  
ann_obj_id = 1 
# Agreguemos un click en (x, y) = (210, 350) para empezar
points = np.array([[210, 350]], dtype=np.float32)
# En las etiquetas, `1` significa positivo `0` significa negativo
labels = np.array([1], np.int32)
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

# Veamos el resultado en el fotograma
plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits[0] > 0.0).cpu().numpy().tolist(), plt.gca(), obj_id=out_obj_ids[0])
plt.axis("off")

### Paso 2: Agrega clic adicionales para refinar la predicción
Parece que, aunque queríamos segmentar la estatua del bobo, el modelo predice la máscara incluyendo partes de la silla e ignorando partes del libro; esto puede suceder ya que existe ambigüedad sobre cuál debería ser el objeto a segmentar si se considera un solo clic. Podemos refinar la máscara en esta imagen mediante otro clic positivo en el libro de la estatua y un clic negativo en la silla.

Aquí hacemos un **segundo clic positivo** en (x, y) = (370, 700) con la etiqueta `1` para expandir la máscara y un **tercer clic negativo** en (x, y) = (370, 1100) con la etiqueta `0` para ignorar esta región de la máscara.

Nota: necesitamos enviar **todos los clics y sus etiquetas** (es decir, no solo los clics adicionales) al llamar la función `add_new_points_or_box`.

In [ ]:
ann_frame_idx = 0  # Indice del primer fotograma
ann_obj_id = 1  # Indice del objeto que queremos segmentar, en este caso es el bobo y se le asignará 1

points = np.array([[210, 350], [370, 700], [370, 1100]], dtype=np.float32)
labels = np.array([1, 1, 0], np.int32)

_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

# Veamos los resultados de segmentación en el primer fotograma
plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits[0] > 0.0).cpu().numpy().tolist(), plt.gca(), obj_id=out_obj_ids[0])
plt.axis("off")

### Paso 3: Propagar las indicaciones para que la máscara aparezca en todo el video
Para que la máscara aparezca en todo el video, propagamos las indicaciones usando la API `propagate_in_video`.

In [ ]:
# Se corren los resultados de la propagación y se almacenan en un diccionario
video_segments = {}  # los segmentos contienen los resultados de segmentación por fotograma
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy().tolist()
        for i, out_obj_id in enumerate(out_obj_ids)
    }

    
bobo_masks = [video_segments[i][1] for i in list(video_segments.keys())]

    
# Veamos los resultados de segmentación cada ciertos frames
vis_frame_stride = 30
for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
    plt.figure(figsize=(6, 4))
    plt.title(f"frame {out_frame_idx}")
    plt.imshow(Image.open(os.path.join(video_dir, frame_names[out_frame_idx])))
    plt.axis("off")
    out_mask = bobo_masks[out_frame_idx]
    show_mask(out_mask, plt.gca(), obj_id=1)

# Trabajo práctico

Ahora queremos utilizar lo que hemos aprendido para segmentar la estatua del bobo en el algunos videos que encuentran en la carpeta  `./videos`. Tenga en cuenta que fueron tomados desde distintos ángulos y distancias.

## Tarea 1: Segmentación de la estatua del bobo

Tu tarea es crear una función llamada select_prompts que tome como entrada el nombre de un `video` (la carpeta que contiene los fotogramas en formato JPEG) y devuelva dos elementos: `points` (una lista de coordenadas de los puntos de interés en el primer fotograma) y `labels` (una lista de etiquetas asociadas a cada punto).

Para ayudarte a desarrollar esta función, te brindamos algunos pasos generales que puedes seguir (opcionalmente). Recuerda que hay muchas maneras de abordar este problema, así que sé creativo y experimenta con diferentes enfoques.

### Pasos generales para desarrollar la función

1. **Cargar y explorar los fotogramas del video:**

* Abre la carpeta que contiene los fotogramas en formato JPEG y carga el primer fotograma para comenzar.
* Pista: Usa os.listdir para listar los archivos y asegúrate de ordenarlos adecuadamente para procesar el primer fotograma.

2. **Preparar la imagen para el procesamiento:**

* Convierte la imagen a un espacio de color que facilite la detección de características.
* Pista: Prueba usando el espacio de color HSV, ya que puede ayudarte a distinguir mejor ciertos elementos en la imagen.

3. **Aplicar preprocesamiento a la imagen:**

* Usa técnicas de procesamiento de imágenes para limpiar o resaltar las regiones de interés.
* Pista: Considera usar operaciones morfológicas, como la apertura, para reducir el ruido y definir mejor las áreas que deseas detectar.

4. **Detectar regiones de interés:**

* Identifica las regiones relevantes en la imagen usando técnicas como el análisis de componentes conectados.
* Pista: Puedes usar funciones como cv2.connectedComponentsWithStats para obtener información útil, como el área y la posición de las regiones detectadas.

5. **Filtrar y seleccionar los puntos más importantes:**

* Define un criterio para elegir las regiones más significativas (por ejemplo, las de mayor tamaño).
* Pista: Si las áreas son demasiado pequeñas, es posible que no sean relevantes; establece un umbral mínimo para evitar detectar ruido.

6. **Retornar los puntos seleccionados junto con su label:**

* Asegúrate de que tu función devuelva los puntos de interés contenidos en un numpy array y su labels asociados contenidos en una lista

In [ ]:
def select_prompts(video):
    #Completa la función, además no cambies los nombres de las variables de entrada ni de salida
    return points, labels

### Revisa y guarda tu respuesta 1
Prueba la función mediante la siguiente celda y guarda tu respuesta.

In [ ]:
assert len(select_prompts('video_009')) > 0, 'Fallas en la declaración de la función'
assert type(select_prompts('video_009')[1]) == list, 'Fallas en la declaración de la función'
assert type(select_prompts('video_009')[0]) == np.ndarray, 'Fallas en la declaración de la función'
assert lm.verify_and_save(1, select_prompts, codigo)== None, 'Fallas de implementación de la función'

### A contiuación observa los resultados cualitativos de la función 1

En la siguiente celda revisa los resultados cualitativos de los puntos predichos por tu función. Estos resultados serán importantes a la hora de evaluar el desempeño de tu método propuesto.

In [ ]:
video = 'video_008'
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

points, labels = select_prompts(video)

plt.figure(figsize=(9, 6))
plt.title(f"frame 0")
plt.imshow(Image.open(os.path.join('videos', video, frame_names[0])))
show_points(np.array(points), np.array(labels), plt.gca())
plt.axis("off")



## Tarea 2: Segmentación de la estatua del bobo

A continuación, deberás crear una función llamada `track_object_videos` que tome como argumento el nombre de un video específico y utilice la función `select_prompts` previamente implementada para segmentar automáticamente la estatua del bobo en ese video. El resultado debe ser un diccionario donde la llave es el nombre del video, y el valor es una lista que contiene las máscaras de la estatua del bobo en cada fotograma de dicho video.

Para ayudarte en el desarrollo de la función, te proporcionamos una serie de pasos opcionales que puedes seguir. Ten en cuenta que hay varias formas de lograr esto, así que siéntete libre de ajustar tu enfoque según sea necesario.

### Pasos opcionales para desarrollar la función:
1. **Cargar el video desde el directorio:**

* Abre el directorio ./videos y busca el video cuyo nombre se corresponde con el argumento proporcionado.
* Pista: Usa os.listdir para verificar la existencia del video y asegurarte de que se encuentra correctamente.

2. **Ordenar y cargar los fotogramas:**

* Ordena los nombres de los fotogramas en orden numérico para asegurar el procesamiento correcto.

3. **Inicializar el predictor:**

* Configura el estado inicial de inferencia utilizando la función de predictor que te permite realizar esta tarea.

4. **Seleccionar puntos de interés en el primer fotograma:**

* Identifica y etiqueta los puntos relevantes en el primer fotograma del video.

5. **Propagar la segmentación a lo largo del video:**

* Utiliza las funciones de predictor necesarias para propagar las máscaras a lo largo de todos los fotogramas.

6. **Almacenar los resultados:**

* Guarda las máscaras de cada fotograma en una lista y a su vez guarda la lista dentro del diccionario output con el nombre del video como llave. Recuerda que sólo son necesarias las máscaras asociadas al bobo, es decir aquellas con obj_id = 1 según las convenciones empleadas anteriormente.


In [ ]:
def track_object_videos(video):
    output = {video: []}
    # Completa la función, por favor no cambies los nombres de las variables de entrada y de salida
    return output

  ### Revisa y guarda tu respuesta 2
Prueba la segunda función usando la siguiente celda y guarda tu respuesta. (Podras ver al verificador de respuestas probando tu función)

In [ ]:
assert lm.verify_and_save(2, track_object_videos, codigo) == None, 'Fallos en la implementación de la función'

### A continuación observa los resultados cualitativos de la función 2

En la siguiente celda revisa los resultados de las máscaras predichas empleando ambas funciones. Estos resultados serán importantes a la hora de evaluar el desempeño de tu método propuesto.

In [ ]:
video_test = 'video_002'
video_dir = os.path.join('videos', video_test)
pred_mask = track_object_videos(video_test)[video_test]
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))


vis_frame_stride = 30
for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
    plt.figure(figsize=(6, 4))
    plt.title(f"frame {out_frame_idx}")
    plt.imshow(Image.open(os.path.join(video_dir, frame_names[out_frame_idx])))
    plt.axis("off")
    show_mask(np.array(pred_mask[out_frame_idx]), plt.gca(), obj_id = 1)



### Ahora evalua el desempeño de tu método propuesto
En la siguiente celda de guardado de respuestas, se evaluará la capacidad de tu abordaje para segmentar la estatua del bobo. En ese sentido, se evaluarán las máscaras predichas y, si tu método es lo suficientemente competente, se te brindará la totalidad del puntaje de la rúbrica. Ten en cuenta que en caso de que esto no sea así, deberás hacer mejoras en el extractor de puntos automático, ya que probablemente los puntos extraídos estén limitando la capacidad de SAM para segmentar. 

### Revisa y guarda tu respuesta 3
Ten en cuenta que esta celda evalua tu método propuesto, por lo que a mayor calidad de las máscaras predichas, más posibilidades hay de que obtengas el puntaje completo

In [ ]:
assert lm.verify_and_save(3, track_object_videos, codigo) == None, 'Fallos en la implementación de la función'

### Guarda todas tus respuestas y preparate para el envío
Corre la siguiente celda para guardar todas tus respuestas y sigue el tutorial de envío del laboratorio

In [ ]:
lm.save_answers(codigo)